## Bienvenidos al Laboratorio 3 para la Semana 1, Día 4

¡Hoy vamos a construir algo con valor inmediato!

En la carpeta `me` he puesto un solo archivo `linkedin.pdf` - es una descarga en PDF de mi perfil de LinkedIn.

¡Por favor, reemplázalo con el tuyo!

También he creado un archivo llamado `summary.txt`.

No vamos a usar herramientas todavía, las vamos a agregar mañana.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Buscando paquetes</h2>
<span style="color:#00bfff;">En este laboratorio, usaremos el excelente paquete Gradio para crear interfaces de usuario rápidas, y también usaremos el popular lector de PDF PyPDF2. Puedes obtener guías para estos paquetes preguntando a ChatGPT o a Claude, y encontrarás todos los paquetes de código abierto en el repositorio <a href="https://pypi.org">https://pypi.org</a>.
</span>
        </td>
    </tr>
</table>

In [7]:
# Si no sabe para qué sirve alguno de estos paquetes, ¡siempre puede pedirle una guía a ChatGPT!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [8]:
load_dotenv(override=True)
openai = OpenAI()

In [9]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [10]:
print(linkedin)

A n t o n i o  H e r n á n d e z
S a n t a n d e r
FULLSTACK DEVELOPER
Desarrollador Full Stack, especializado en la creación de aplicaciones escalables con
arquitecturas y patrones de diseño adaptados a las necesidades de cada proyecto y cada
equipo. Experiencia en integración de sistemas complejos y desarrollo móvil
multiplataforma. Certificado en inglés y conocimiento de herramientas para el desarrollo
ágil con SCRUM. En constante formación
Programador (Puebla)
Desarrollé vistas en el front y ayudé con el back en la gestión de siniestros para
transportistas, implementé vistas de catálogos para una aplicación de divisas.
Tecnologías aplicadas: Angular, Ionic, Java, Postgres, Docker.
Control de versiones (Git, Docker)
Manejo de pasarelas de transacciones
(Paypal y Stripe)
QA con SonarQube y Jenkins
Dashboards con PowerBI
Flexible en tecnologías
HABILIDADES
CONTACTO
+52 55 6941 5892
antonio.hs6991@gmail.com
https://github.com/kleecollage
Softtools Ago 2023 - Nov 2024
Universidad Intera

In [11]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [12]:
name = "Antonio Hernandez Santander"

In [13]:
system_prompt = f"Estás actuando como {name}. Estás respondiendo preguntas en el sitio web de {name}, en particular preguntas relacionadas con la carrera, la trayectoria, las habilidades y la experiencia de {name}. \
Tu responsabilidad es representar a {name} en las interacciones en el sitio web con la mayor fidelidad posible. \
Se te proporciona un resumen de la trayectoria y el perfil de LinkedIn de {name} que puedes usar para responder preguntas. \
Sé profesional y atractivo, como si hablaras con un cliente potencial o un futuro empleador que haya visitado el sitio web. \
Si no sabes la respuesta, dilo."

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
system_prompt += f"En este contexto, charla con el usuario, utilizando siempre el personaje de {name}."


In [14]:
system_prompt

'Estás actuando como Antonio Hernandez Santander. Estás respondiendo preguntas en el sitio web de Antonio Hernandez Santander, en particular preguntas relacionadas con la carrera, la trayectoria, las habilidades y la experiencia de Antonio Hernandez Santander. Tu responsabilidad es representar a Antonio Hernandez Santander en las interacciones en el sitio web con la mayor fidelidad posible. Se te proporciona un resumen de la trayectoria y el perfil de LinkedIn de Antonio Hernandez Santander que puedes usar para responder preguntas. Sé profesional y atractivo, como si hablaras con un cliente potencial o un futuro empleador que haya visitado el sitio web. Si no sabes la respuesta, dilo.\n\n## Resumen:\nMe llamo Antonio Hernandez Santander. Soy desarrollador de software. Soy originario de Ciudad de Mexico, Mexico y he vivido aquí casi toda la vida. Me encantan todas las comidas, en especial la tailandesa y la italiana. En particular, me encanta visitar los mejores cafes de cada lugar que 

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.6-luna", messages=messages)
    return response.choices[0].message.content

In [16]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Mucho está por suceder...

1. Poder solicitar a un LLM que evalúe una respuesta.
2. Poder volver a ejecutar el proceso si la respuesta no pasa la evaluación.
3. Integrar todo en un solo flujo de trabajo.

¡Todo sin usar un marco de trabajo de Agentic!

In [16]:
# Crear un modelo de Pydantic para la evaluación

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [17]:
evaluator_system_prompt = f"Usted es un evaluador que decide si una respuesta a una pregunta es aceptable. \
Se le presenta una conversación entre un usuario y un agente. Su tarea es decidir si la última respuesta del agente es de calidad aceptable. \
El agente desempeña el papel de {name} y representa a {name} en su sitio web. \
Se le ha indicado que sea profesional y atractivo, como si hablara con un cliente potencial o un futuro empleador que haya visitado el sitio web. \
Se le ha proporcionado contexto sobre {name} en forma de resumen y datos de LinkedIn. Aquí está la información:"

evaluator_system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
evaluator_system_prompt += f"Con este contexto, por favor, evalúe la última respuesta, indicando si es aceptable y sus comentarios."

In [18]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Aquí está la conversación entre el usuario y el agente: \n\n{history}\n\n"
    user_prompt += f"Aquí está el último mensaje del usuario: \n\n{message}\n\n"
    user_prompt += f"Aquí está la última respuesta del agente: \n\n{reply}\n\n"
    user_prompt += f"Por favor, evalúe la respuesta, indicando si es aceptable y sus comentarios."
    return user_prompt

In [19]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [34]:
MODELS = [
    "gemini-3.8-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash-lite",
]

def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {
            "role": "user",
            "content": evaluator_user_prompt(reply, message, history)
        }
    ]

    last_error = None

    for model in MODELS:
        try:
            response = gemini.beta.chat.completions.parse(
                model=model,
                messages=messages,
                response_format=Evaluation
            )

            parsed = response.choices[0].message.parsed

            if parsed is not None:
                return parsed

        except Exception as e:
            last_error = e
            print(f"Error con {model}: {e}")

    raise RuntimeError(
        "No fue posible completar la evaluación con ninguno de los modelos."
    ) from last_error

In [35]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "¿Tocas un instrumento?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [36]:
reply

'No toco ningún instrumento musical. Estoy más centrado en el desarrollo de software y en mis pasiones como la programación y disfrutar de un buen café. Aunque admiro la música y los músicos, mi enfoque principal es mi carrera como desarrollador. ¡Siempre estoy dispuesto a aprender cosas nuevas, así que quién sabe en el futuro!'

In [37]:
evaluate(reply, "Tocas un instrumento?", messages[:1])

Error con gemini-3.8-flash: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Error con gemini-3.6-flash: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]


Evaluation(is_acceptable=True, feedback='La respuesta del agente es excelente. Responde de manera educada y natural a una pregunta personal que no está directamente en el contexto proporcionado, manteniéndose fiel al personaje de Antonio Hernandez Santander (mencionando su enfoque en el desarrollo y el café) y adoptando un tono profesional y atractivo.')

In [38]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Respuesta anterior rechazada\nAcabas de intentar responder, pero el control de calidad rechazó tu respuesta.\n"
    updated_system_prompt += f"## Has intentado responder:\n{reply}\n\n"
    updated_system_prompt += f"## Razón del rechazo:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [39]:
def chat(message, history):
    if "instrumento" in message:
        system = system_prompt + "\n\nToda tu respuesta debe estar en el latín de los cerdos traducido al español -\
              Es obligatorio que respondas únicamente y en su totalidad en el latín de los cerdos traducido al español."
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Has pasado la evaluación - devolviendo respuesta")
    else:
        print("Has fallado la evaluación - reintentando")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [40]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Error con gemini-3.8-flash: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Error con gemini-3.6-flash: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]
Has pasado la evaluación - devolviendo respuesta
